In [27]:
class DSU:
    """Система непересекающихся множеств (DSU) с ранговой эвристикой"""
    def __init__(self, n):
        self.p, self.r = list(range(n)), [0] * n  # родитель и ранг
    def find(self, x):
        """Поиск корня с сжатием пути"""
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, x, y):
        """Объединение множеств. Возвращает True если объединили, False если уже в одном"""
        x, y = self.find(x), self.find(y)
        if x == y: return False
        # объединяем по рангу: меньший корень подвешиваем к большему
        if self.r[x] < self.r[y]: x, y = y, x
        self.p[y] = x
        if self.r[x] == self.r[y]: self.r[x] += 1
        return True

def kruskal(n, edges):
    """Алгоритм Краскала: возвращает рёбра MST (минимального остова)"""
    d = DSU(n)
    # сортируем по весу и добавляем рёбра, не создающие цикл
    return [e for e in sorted(edges, key=lambda e: e[2]) if d.union(e[0], e[1])]

def bot(t):
    """Максимальный вес на ребре в остове (bottleneck)"""
    return max(w for _, _, w in t)

def cost(t):
    """Суммарный вес остова"""
    return sum(w for _, _, w in t)

def is_span(n, es):
    """Проверяет, является ли набор рёбер остовным деревом на n вершинах"""
    if len(es) != n - 1: return False
    d = DSU(n)
    return all(d.union(u, v) for u, v, w in es)

from itertools import combinations

def min_bot(n, edges):
    """Минимально возможный bottleneck среди всех остовов (перебором)"""
    return min(bot(list(s)) for s in combinations(edges, n-1) if is_span(n, list(s)))

# Пример графа: треугольник с весами (0-1:1, 1-2:2, 0-2:2)
n = 3
E = [(0, 1, 1), (1, 2, 2), (0, 2, 2)]

# MST от Краскала: возьмёт (0,1,1) и (1,2,2) или (0,2,2) — зависит от порядка,
# но из-за сортировки возьмёт первое ребро 1, потом по циклу пропустит 2-е?, уточним:
# на самом деле edges = [(0,1,1),(1,2,2),(0,2,2)] -> после сортировки: сначала (0,1,1),
# потом (1,2,2) — добавит, потом (0,2,2) — уже в одном компоненте, пропустит.
T = kruskal(n, E)

# Альтернативный остов (не MST, но MBST): рёбра (0-2,2) и (1-2,2)
alt = [(0, 2, 2), (1, 2, 2)]

print(f"MST : {T}  cost={cost(T)}  bot={bot(T)}")
print(f"Alt : {alt}  cost={cost(alt)}  bot={bot(alt)}")
print(f"(a) alt — тоже MBST (bot={bot(alt)}), но не MST (cost {cost(alt)} > {cost(T)}): контрпример")
print(f"(b) bot(MST)={bot(T)} == min_bot={min_bot(n, E)}: MST всегда является MBST")

MST : [(0, 1, 1), (1, 2, 2)]  cost=3  bot=2
Alt : [(0, 2, 2), (1, 2, 2)]  cost=4  bot=2
(a) alt — тоже MBST (bot=2), но не MST (cost 4 > 3): контрпример
(b) bot(MST)=2 == min_bot=2: MST всегда является MBST


In [19]:
from collections import defaultdict
# a
def mst_stays_minimal(mst_edges, v, w, c):
    # Строим список смежности
    adj = defaultdict(list)
    for u1, v1, wt in mst_edges:
        adj[u1].append((v1, wt))
        adj[v1].append((u1, wt))

    # Поиск максимального веса на пути v -> w через DFS
    def dfs(current, target, visited, max_edge):
        if current == target:
            return max_edge
        visited.add(current)
        for neighbor, weight in adj[current]:
            if neighbor not in visited:
                res = dfs(neighbor, target, visited, max(max_edge, weight))
                if res is not None:
                    return res
        return None

    max_on_path = dfs(v, w, set(), 0)

    return c >= max_on_path

mst = [
    (0, 1, 2),
    (1, 2, 3),
    (2, 3, 4)
]

print(mst_stays_minimal(mst, 0, 3, 5))
print(mst_stays_minimal(mst, 0, 3, 3))

# b
def update_mst(mst_edges, v, w, c):
    adj = {}
    for a, b, wt in mst_edges:
        adj.setdefault(a, {})[b] = wt
        adj.setdefault(b, {})[a] = wt

    # 2. DFS ищет путь v -> w и запоминает рёбра на пути
    def find_path(cur, target, visited, path_edges):
        if cur == target:
            return True
        visited.add(cur)
        for nb, wt in adj.get(cur, {}).items():
            if nb not in visited:
                path_edges.append((cur, nb, wt))
                if find_path(nb, target, visited, path_edges):
                    return True
                path_edges.pop()
        return False

    path_edges = []
    find_path(v, w, set(), path_edges)

    # Находим самое тяжёлое ребро на пути
    max_edge = max(path_edges, key=lambda e: e[2])

    # Строим новый MST: удаляем max_edge, добавляем новое
    new_mst = []
    for a, b, wt in mst_edges:
        if {a, b} != {max_edge[0], max_edge[1]}:
            new_mst.append((a, b, wt))
    new_mst.append((v, w, c))
    return new_mst


old_mst = [(0, 1, 10), (1, 2, 20), (2, 3, 30)]
new = update_mst(old_mst, 0, 3, 15)
print(new)

True
False
[(0, 1, 10), (1, 2, 20), (0, 3, 15)]


In [29]:
from itertools import groupby, permutations

class DSU:
    """Система непересекающихся множеств"""
    def __init__(self, n):
        self.p, self.r = list(range(n)), [0] * n
    def find(self, x):
        if self.p[x] != x: self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, x, y):
        x, y = self.find(x), self.find(y)
        if x == y: return False
        if self.r[x] < self.r[y]: x, y = y, x
        self.p[y] = x
        if self.r[x] == self.r[y]: self.r[x] += 1
        return True

def kruskal(n, edges):
    """Классический Краскал: сортировка по весу"""
    d = DSU(n)
    return [e for e in sorted(edges, key=lambda e: e[2]) if d.union(e[0], e[1])]

def cost(t):
    """Суммарный вес остова"""
    return sum(w for _, _, w in t)

def run(n, order):
    """Запуск Краскала с заданным порядком рёбер (без сортировки)"""
    d = DSU(n)
    # добавляем рёбра строго в порядке order, пока не наберём остов
    t = [e for e in order if d.union(e[0], e[1])]
    # нормализуем: сортируем вершины внутри ребра и сами рёбра
    return tuple(sorted((min(u,v), max(u,v), w) for u, v, w in t))

def all_mst(n, edges):
    """
    Генерация ВСЕХ минимальных остовных деревьев (MST)
    Идея: перебираем все перестановки рёбер внутри каждой группы одинакового веса
    """
    # группируем рёбра по весу
    grps = [list(g) for _, g in groupby(sorted(edges, key=lambda e: e[2]), key=lambda e: e[2])]
    
    # строим все возможные порядки: внутри каждой группы все перестановки
    orders = [[]]
    for g in grps:
        orders = [o + list(p) for o in orders for p in permutations(g)]
    
    seen, res = set(), []
    for o in orders:
        t = run(n, o)
        if len(t) == n-1 and t not in seen:
            seen.add(t)
            res.append(list(t))
    return res

def order_for(n, edges, T):
    """
    Строит порядок рёбер, при котором Краскал выдаст заданное MST 'T'
    Рёбра из T идут первыми внутри своей весовой группы
    """
    ts = {(min(u,v), max(u,v)) for u, v, w in T}
    # сортируем: сначала по весу, затем рёбра из T идут раньше (0) чем не из T (1)
    return sorted(edges, key=lambda e: (e[2], 0 if (min(e[0],e[1]), max(e[0],e[1])) in ts else 1))

# Пример: граф с кратными рёбрами веса 1 и 2
n = 4
E = [(0,1,1), (1,2,1), (0,2,1), (2,3,2), (1,3,2)]

# Строим одно MST через обычный Краскал
T = kruskal(n, E)
T_canon = run(n, T)  # канонический вид MST

# Проверяем: существует ли порядок, при котором Краскал даст именно это MST?
reproduced = run(n, order_for(n, E, T))

print(f"MST T: {sorted(T)}")
print(f"Kruskal с построенным порядком воспроизводит T: {reproduced == T_canon}")

# Находим ВСЕ возможные MST в этом графе
trees = all_mst(n, E)
print(f"\nВсе MST ({len(trees)} шт.), cost={cost(trees[0])} у каждого:")
for t in trees:
    print(f"  {sorted(t)}")

MST T: [(0, 1, 1), (1, 2, 1), (2, 3, 2)]
Kruskal с построенным порядком воспроизводит T: True

Все MST (6 шт.), cost=4 у каждого:
  [(0, 1, 1), (1, 2, 1), (2, 3, 2)]
  [(0, 1, 1), (1, 2, 1), (1, 3, 2)]
  [(0, 1, 1), (0, 2, 1), (2, 3, 2)]
  [(0, 1, 1), (0, 2, 1), (1, 3, 2)]
  [(0, 2, 1), (1, 2, 1), (2, 3, 2)]
  [(0, 2, 1), (1, 2, 1), (1, 3, 2)]
